In [2]:
from dotenv import load_dotenv
import os
import json
import replicate
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import gradio as gr
from typing import List, Dict, Tuple
import textwrap
import time
import re

# For LLM integration - install: pip install groq (or openai)
try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

load_dotenv()

class IslamicComicGenerator:
    def __init__(self, replicate_token: str = None, llm_provider: str = "groq"):
        """
        Initialize with Replicate token and LLM provider ('groq', 'openai', or 'none')
        """
        # Replicate setup
        self.replicate_token = replicate_token or os.getenv("REPLICATE_API_TOKEN")
        if not self.replicate_token:
            raise ValueError("REPLICATE_API_TOKEN required")
        
        os.environ["REPLICATE_API_TOKEN"] = self.replicate_token
        self.client = replicate.Client(api_token=self.replicate_token)
        
        # LLM setup
        self.llm_provider = llm_provider.lower()
        self.llm_client = None
        
        if self.llm_provider == "groq" and GROQ_AVAILABLE:
            api_key = os.getenv("GROQ_API_KEY")
            if api_key:
                self.llm_client = Groq(api_key=api_key)
                print("✓ Using Groq LLM (Llama 3.3 70B)")
            else:
                print("⚠️ Groq API key not found, falling back to rule-based")
                self.llm_provider = "none"
                
        elif self.llm_provider == "openai" and OPENAI_AVAILABLE:
            api_key = os.getenv("OPENAI_API_KEY")
            if api_key:
                self.llm_client = OpenAI(api_key=api_key)
                print("✓ Using OpenAI GPT-4o-mini")
            else:
                print("⚠️ OpenAI API key not found, falling back to rule-based")
                self.llm_provider = "none"
        else:
            print("ℹ️ Using rule-based scene analysis (no LLM)")
            self.llm_provider = "none"
        
        # Style settings
        self.style_suffix = ", Islamic manuscript illumination style, flat perspective, rich textures, comic book panel, geometric arabesque borders, dramatic lighting"
        self.standard_negative = "human face, portrait, realistic person, photorealistic human, figure, man, woman, child, body, hands, modern clothing, contemporary architecture, western perspective, 3d render, anime, photographic"
    
    def analyze_scene_with_llm(self, text: str) -> Dict:
        """
        Use LLM to intelligently analyze text and extract scenes with proper prompts
        Returns structured data matching the old format for compatibility
        """
        system_prompt = """You are an Islamic Visual Narrative Designer. Convert Islamic stories into structured image generation prompts.

CRITICAL RULES:
1. NEVER describe prophets' faces, bodies, or forms (Muhammad, Moses, Jesus, Abraham, Noah, etc.)
2. If a scene centers on a prophet, use these alternatives:
   - "first_person": What the prophet sees (POV shot)
   - "environment": The setting where it happened
   - "reaction": Other people watching (backs turned, no faces)
   - "object_focus": Close-up of significant objects (burning bush, ark, tablets)
3. Determine the historical setting accurately
4. Create vivid, detailed image prompts (50-80 words) focusing on atmosphere, lighting, and environment

Output must be valid JSON with this structure:
{
    "scene_type": "environment" | "object_focus" | "reaction" | "first_person",
    "setting": "desert" | "ancient_egypt" | "mountain" | "ocean" | "valley",
    "image_prompt": "detailed description for image generation...",
    "reasoning": "brief explanation of why this scene type was chosen"
}"""

        user_prompt = f"""Analyze this Islamic story segment and create an image generation prompt:

STORY SEGMENT:
{text}

Remember: No human figures if prophets are present. Focus on environment, objects, or reactions from behind."""

        try:
            if self.llm_provider == "groq":
                response = self.llm_client.chat.completions.create(
                    model="llama-3.3-70b-versatile",  # or "mixtral-8x7b-32768" for faster/cheaper
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.7,
                    max_tokens=500,
                    response_format={"type": "json_object"}
                )
                result = json.loads(response.choices[0].message.content)
                
            elif self.llm_provider == "openai":
                response = self.llm_client.chat.completions.create(
                    model="gpt-4o-mini",  # or "gpt-3.5-turbo" for cheaper
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.7,
                    max_tokens=500,
                    response_format={"type": "json_object"}
                )
                result = json.loads(response.choices[0].message.content)
            else:
                raise Exception("No LLM provider")
            
            # Validate and return in expected format
            return {
                "scene_type": result.get("scene_type", "environment"),
                "contains_prophet": "prophet" in text.lower() or result.get("scene_type") in ["first_person", "object_focus"],
                "settings": [result.get("setting", "desert")],
                "text": text,
                "llm_prompt": result.get("image_prompt", ""),
                "reasoning": result.get("reasoning", "")
            }
            
        except Exception as e:
            print(f"  LLM failed ({e}), falling back to rule-based")
            return self._analyze_scene_rule_based(text)
    
    def _analyze_scene_rule_based(self, text: str) -> Dict:
        """Fallback rule-based analysis if LLM fails"""
        text_lower = text.lower()
        
        prophets = ["muhammad", "moses", "musa", "abraham", "ibrahim", "noah", "nuh", 
                   "jesus", "isa", "joseph", "yusuf", "david", "dawud", "solomon", "sulayman"]
        contains_prophet = any(prophet in text_lower for prophet in prophets)
        
        if any(word in text_lower for word in ["bush", "ark", "tablet", "scroll", "staff", "throne"]):
            scene_type = "object_focus"
        elif any(word in text_lower for word in ["said", "spoke", "asked", "replied", "crowd", "people"]):
            scene_type = "reaction" if contains_prophet else "environment"
        elif any(word in text_lower for word in ["saw", "behold", "looked"]):
            scene_type = "first_person"
        else:
            scene_type = "environment"
        
        settings = []
        if any(w in text_lower for w in ["desert", "sinai"]): settings.append("desert")
        if any(w in text_lower for w in ["nile", "egypt", "pharaoh"]): settings.append("ancient_egypt")
        if any(w in text_lower for w in ["mountain", "cave"]): settings.append("mountain")
        if any(w in text_lower for w in ["ocean", "sea", "flood", "ark"]): settings.append("ocean")
        if any(w in text_lower for w in ["valley", "tuwa"]): settings.append("valley")
        
        return {
            "scene_type": scene_type,
            "contains_prophet": contains_prophet,
            "settings": settings if settings else ["desert"],
            "text": text,
            "reasoning": "rule-based fallback"
        }
    
    def build_prompt(self, scene_analysis: Dict) -> Dict:
        """Build final prompt - use LLM prompt if available, otherwise template"""
        scene_type = scene_analysis["scene_type"]
        
        # If LLM provided a custom prompt, use it (but add style suffix)
        if "llm_prompt" in scene_analysis and scene_analysis["llm_prompt"]:
            main_prompt = scene_analysis["llm_prompt"] + self.style_suffix
        else:
            # Fallback to templates
            settings = scene_analysis["settings"]
            text = scene_analysis["text"]
            
            setting_desc = {
                "desert": "vast desert landscape, sand dunes, golden hour lighting",
                "ancient_egypt": "ancient Egyptian setting, Nile river, mud-brick architecture",
                "mountain": "rugged mountain terrain, rocky slopes, cave entrance",
                "ocean": "vast ocean, wooden ship construction, storm clouds",
                "valley": "narrow valley, lush vegetation, protective rock formations"
            }
            main_setting = setting_desc.get(settings[0], setting_desc["desert"])
            
            if scene_type == "object_focus":
                main_prompt = self._object_focus_template(main_setting, text) + self.style_suffix
            elif scene_type == "reaction":
                main_prompt = self._reaction_template(main_setting, text) + self.style_suffix
            elif scene_type == "first_person":
                main_prompt = self._first_person_template(main_setting, text) + self.style_suffix
            else:
                main_prompt = self._environment_template(main_setting, text) + self.style_suffix
        
        return {
            "main_prompt": main_prompt,
            "negative_prompt": self.standard_negative,
            "scene_type": scene_type,
            "original_text": scene_analysis["text"][:100] + "...",
            "reasoning": scene_analysis.get("reasoning", "template")
        }
    
    # Template methods remain the same...
    def _environment_template(self, setting: str, text: str) -> str:
        return f"{setting}, wide panoramic view, no human figures in foreground, dramatic sky, earth tones"
    
    def _object_focus_template(self, setting: str, text: str) -> str:
        text_lower = text.lower()
        if "bush" in text_lower:
            object_desc = "miraculous burning bush, flames that do not consume"
        elif "ark" in text_lower:
            object_desc = "massive wooden ark under construction, cedar wood beams"
        else:
            object_desc = "sacred ancient artifact, divine light emanating"
        return f"Close-up of {object_desc}, {setting}, intricate details, dramatic lighting"
    
    def _reaction_template(self, setting: str, text: str) -> str:
        return f"Group of robed figures viewed from behind, looking toward empty space, {setting}, Islamic miniature style"
    
    def _first_person_template(self, setting: str, text: str) -> str:
        return f"First-person perspective view, {setting}, looking at miraculous scene, divine light in distance"
    
    def generate_image(self, prompt_data: Dict, width: int = 1024, height: int = 768) -> Image.Image:
        """Generate image using Replicate"""
        try:
            print(f"  Calling Imagen-4...")
            output = self.client.run(
                "google/imagen-4",
                input={
                    "prompt": prompt_data["main_prompt"],
                    "aspect_ratio": "4:3",
                }
            )
            
            # Handle FileOutput object or list
            image_url = None
            if isinstance(output, list) and len(output) > 0:
                image_url = output[0]
            elif isinstance(output, str):
                image_url = output
            elif hasattr(output, 'url'):
                image_url = output.url
            else:
                urls = re.findall('https?://[^\s<>\'\"]+', str(output))
                if urls:
                    image_url = urls[0]
            
            if not image_url:
                raise ValueError(f"Could not extract URL from {type(output)}")
            
            response = requests.get(image_url, timeout=30)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content))
            
            if img.size != (width, height):
                img = img.resize((width, height), Image.Resampling.LANCZOS)
            
            return img
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            img = Image.new('RGB', (width, height), color='#F5E6D3')
            draw = ImageDraw.Draw(img)
            try:
                font = ImageFont.truetype("Arial.ttf", 24)
            except:
                font = ImageFont.load_default()
            draw.text((width//2, height//2), f"Error:\n{str(e)[:50]}", 
                     fill='#8B4513', font=font, anchor="mm")
            return img
    
    def create_comic_page(self, story_text: str, panels_per_row: int = 2) -> Tuple[Image.Image, List[Dict]]:
        """
        New workflow: Use LLM to first split story into scenes, then generate each
        """
        print("=" * 60)
        print("Step 1: Analyzing story with LLM...")
        
        # Step 1: Use LLM to break story into scenes and create prompts
        scenes = self._extract_scenes_with_llm(story_text)
        print(f"✓ LLM extracted {len(scenes)} scenes")
        
        # Step 2: Generate images for each scene
        panels = []
        panel_data = []
        
        print(f"\nStep 2: Generating {len(scenes)} images...")
        for i, scene in enumerate(scenes):
            print(f"\nPanel {i+1}/{len(scenes)}: {scene['scene_type'].upper()}")
            if 'reasoning' in scene:
                print(f"  Reasoning: {scene['reasoning'][:60]}...")
            
            # Build final prompt (combines LLM output with templates)
            prompt_data = self.build_prompt(scene)
            print(f"  Prompt: {prompt_data['main_prompt'][:60]}...")
            
            img = self.generate_image(prompt_data)
            panels.append(img)
            panel_data.append(prompt_data)
            
            # Rate limiting
            if i < len(scenes) - 1:
                print(f"  ⏳ Waiting 10s...")
                time.sleep(10)
        
        # Step 3: Assemble
        print(f"\nStep 3: Assembling comic page...")
        comic = self._assemble_panels(panels, panel_data, panels_per_row)
        return comic, panel_data
    
    def _extract_scenes_with_llm(self, text: str) -> List[Dict]:
        """
        Use LLM to intelligently split story into visual scenes
        """
        if self.llm_provider == "none":
            # Fallback: split by sentences
            sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 10]
            return [self._analyze_scene_rule_based(s) for s in sentences[:4]]
        
        system_prompt = """You are an expert at converting narrative text into visual storyboard scenes.

Task: Break down the following Islamic story into 2-4 distinct visual scenes suitable for a comic book.

For each scene, determine:
1. scene_type: How to handle prophet depiction (first_person, environment, reaction, object_focus)
2. setting: The location (desert, ancient_egypt, mountain, ocean, valley)
3. image_prompt: Detailed visual description (avoid human figures if prophets present)
4. text: The specific text segment this scene represents

CRITICAL: Respect Islamic guidelines - no prophet faces/bodies.

Output valid JSON array:
[
  {
    "scene_type": "environment",
    "setting": "valley",
    "image_prompt": "Detailed description...",
    "text": "Original story segment..."
  }
]"""

        try:
            if self.llm_provider == "groq":
                response = self.llm_client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": f"Story:\n{text}\n\nCreate 2-4 visual scenes:"}
                    ],
                    temperature=0.7,
                    max_tokens=2000,
                    response_format={"type": "json_object"}
                )
                # Groq might return {"scenes": [...]} or direct array
                content = json.loads(response.choices[0].message.content)
                if isinstance(content, list):
                    scenes = content
                elif "scenes" in content:
                    scenes = content["scenes"]
                else:
                    scenes = [content]  # Single scene
                    
            elif self.llm_provider == "openai":
                response = self.llm_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": f"Story:\n{text}"}
                    ],
                    temperature=0.7,
                    max_tokens=2000,
                    response_format={"type": "json_object"}
                )
                content = json.loads(response.choices[0].message.content)
                scenes = content if isinstance(content, list) else content.get("scenes", [content])
            
            # Validate and clean
            valid_scenes = []
            for scene in scenes[:4]:  # Max 4 panels
                valid_scenes.append({
                    "scene_type": scene.get("scene_type", "environment"),
                    "setting": scene.get("setting", "desert"),
                    "text": scene.get("text", ""),
                    "llm_prompt": scene.get("image_prompt", ""),
                    "contains_prophet": scene.get("scene_type") in ["first_person", "object_focus"],
                    "settings": [scene.get("setting", "desert")],
                    "reasoning": "LLM analyzed"
                })
            return valid_scenes
            
        except Exception as e:
            print(f"  LLM scene extraction failed: {e}")
            # Fallback
            sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 10]
            return [self._analyze_scene_rule_based(s) for s in sentences[:4]]
    
    def _assemble_panels(self, panels: List[Image.Image], panel_data: List[Dict], panels_per_row: int) -> Image.Image:
        """Assemble panels into comic grid"""
        panel_width = 1024
        panel_height = 768
        padding = 20
        caption_height = 120
        
        rows = (len(panels) + panels_per_row - 1) // panels_per_row
        page_width = panels_per_row * (panel_width + padding) + padding
        page_height = rows * (panel_height + caption_height + padding) + padding
        
        canvas = Image.new('RGB', (page_width, page_height), '#FAF0E6')
        draw = ImageDraw.Draw(canvas)
        
        try:
            small_font = ImageFont.truetype("Arial.ttf", 16)
        except:
            small_font = ImageFont.load_default()
        
        for idx, (img, data) in enumerate(zip(panels, panel_data)):
            row = idx // panels_per_row
            col = idx % panels_per_row
            
            x = padding + col * (panel_width + padding)
            y = padding + row * (panel_height + caption_height + padding)
            
            canvas.paste(img, (x, y))
            
            # Border
            draw.rectangle([x-2, y-2, x+panel_width+2, y+panel_height+2], 
                          outline='#8B4513', width=3)
            
            # Caption
            caption = data['original_text']
            wrapped = textwrap.fill(caption, width=70)
            draw.text((x, y + panel_height + 10), wrapped, 
                     fill='#4A3728', font=small_font)
            
            # Type label
            draw.text((x, y - 15), f"Type: {data['scene_type']}", 
                     fill='#666666', font=small_font)
        
        return canvas
    
    def interactive_generate(self, story_text: str, panels_per_row: int = 2):
        """Gradio interface"""
        if not story_text or len(story_text) < 20:
            return None, "Please enter a longer story (at least 20 characters)."
        
        try:
            comic_page, panel_data = self.create_comic_page(story_text, panels_per_row)
            output_path = "generated_comic.png"
            comic_page.save(output_path)
            
            # Cost calculation
            img_cost = len(panel_data) * 0.003  # Imagen-4 cost
            if self.llm_provider == "groq":
                llm_cost = 0  # Free tier
            elif self.llm_provider == "openai":
                llm_cost = 0.005  # Approximate for gpt-4o-mini
            else:
                llm_cost = 0
            
            total = img_cost + llm_cost
            return comic_page, f"✓ Generated {len(panel_data)} panels | Cost: ~${total:.3f} (LLM: {self.llm_provider})"
            
        except Exception as e:
            return None, f"Error: {str(e)}"

def main():
    load_dotenv()
    replicate_key = os.getenv("REPLICATE_API_TOKEN")
    
    print("=" * 60)
    print("Islamic Comic Generator with LLM")
    print("=" * 60)
    
    if not replicate_key:
        print("❌ REPLICATE_API_TOKEN not found")
        return
    
    # Choose LLM provider
    print("\nChoose LLM provider:")
    print("1. Groq (Free, fast, llama-3.3-70b)")
    print("2. OpenAI (Reliable, gpt-4o-mini, ~$0.005/story)")
    print("3. None (Rule-based only)")
    
    choice = input("\nSelect (1/2/3): ").strip()
    provider = {"1": "groq", "2": "openai", "3": "none"}.get(choice, "groq")
    
    try:
        generator = IslamicComicGenerator(replicate_key, llm_provider=provider)
    except Exception as e:
        print(f"❌ Initialization failed: {e}")
        return
    
    example = """When Moses approached the burning bush in the valley of Tuwa, he saw flames that burned without consuming the green leaves. A divine light radiated from the bush, illuminating the entire valley. The shepherds stood at a distance, watching in awe as Moses slowly approached the sacred ground."""
    
    print(f"\nExample: {example[:100]}...")
    use_ex = input("Use example? (y/n): ").lower() == 'y'
    
    if use_ex:
        story = example
    else:
        print("\nEnter story (empty line to finish):")
        lines = []
        while True:
            line = input()
            if not line and lines:
                break
            lines.append(line)
        story = " ".join(lines)
    
    if not story.strip():
        return
    
    print(f"\n🔍 Using {provider.upper()} to analyze story...")
    comic, data = generator.create_comic_page(story, panels_per_row=2)
    comic.save("islamic_comic_llm.png")
    print(f"\n✓ Saved to: islamic_comic_llm.png")
    print(f"✓ Generated {len(data)} panels with AI-powered scene analysis")

if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "--web":
        # For web UI, default to Groq if available
        load_dotenv()
        gen = IslamicComicGenerator(llm_provider="groq")
        
        with gr.Blocks(title="Islamic Comic Generator + LLM") as demo:
            gr.Markdown("""# 📖 Islamic Comic Generator with LLM
            **AI-powered scene analysis** using Groq (Llama 3.3) or OpenAI""")
            
            with gr.Row():
                with gr.Column():
                    story = gr.Textbox(label="Story", lines=6, value="Moses approached the burning bush in the valley. Flames danced around the leaves but did not consume them. A divine light illuminated the valley.")
                    panels = gr.Slider(1, 4, value=2, step=1, label="Panels per row")
                    btn = gr.Button("Generate", variant="primary")
                
                with gr.Column():
                    img_out = gr.Image(label="Comic")
                    txt_out = gr.Textbox(label="Status")
            
            btn.click(fn=gen.interactive_generate, inputs=[story, panels], outputs=[img_out, txt_out])
        
        demo.launch()
    else:
        main()

Islamic Comic Generator with LLM

Choose LLM provider:
1. Groq (Free, fast, llama-3.3-70b)
2. OpenAI (Reliable, gpt-4o-mini, ~$0.005/story)
3. None (Rule-based only)
✓ Using Groq LLM (Llama 3.3 70B)

Example: When Moses approached the burning bush in the valley of Tuwa, he saw flames that burned without cons...

🔍 Using GROQ to analyze story...
Step 1: Analyzing story with LLM...
✓ LLM extracted 3 scenes

Step 2: Generating 3 images...

Panel 1/3: ENVIRONMENT
  Reasoning: LLM analyzed...
  Prompt: A serene valley with a prominent bush at its center, surroun...
  Calling Imagen-4...
  ⏳ Waiting 10s...

Panel 2/3: OBJECT_FOCUS
  Reasoning: LLM analyzed...
  Prompt: A close-up of the burning bush, with the flames and the divi...
  Calling Imagen-4...
  ❌ Error: ReplicateError Details:
title: Insufficient credit
status: 402
detail: You have insufficient credit to run this model. Go to https://replicate.com/account/billing#billing to purchase credit. Once you purchase credit, please wait a 